# Leakage-resistant benchmark: source-group LC25000 split × 5 seeds, with external testing on LungHist700
**What this notebook does**
1. Rebuilds the LC25000 lung data (MD5-deduplicated, as before) and attaches each image's **source-tile group**
   from the published LC25000-clean grouping (Batchkala et al., 2025). All augmented copies of one source tile
   are kept in the same partition, so no test image has a sibling in training.
2. For each of **5 seeds**, makes a new group-level 70/15/15 split and trains MobileNetV2, EfficientNetB0, the
   hybrid EfficientNetB0–Transformer and a parameter-matched ablation (same block without attention)
   with the architectures and training protocol of the first benchmark; the only change is source-group partitioning.
3. Tests every trained model on **LungHist700 without any retraining or tuning** (primary analysis: 12 tiles, no normalisation).
4. Reports mean ± SD and 95% CIs across seeds, paired comparisons, calibration, and patient counts.

**How to run (Google Colab):** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.
Allow Google Drive access when asked. Total time is about **5–8 hours** for 5 seeds (about 1–1.5 hours per seed).
Every finished model is saved to Google Drive immediately. **If Colab disconnects, just run all cells again:**
finished runs are skipped automatically and the notebook continues where it stopped.
At the end it prints a **SUMMARY** and downloads **group_split_results.zip**. Send both to Claude.

# Pre-run checklist for the publication-grade experiment

**Do not interpret the rerun as successful merely because accuracy is higher.** Its purpose is to produce a leakage-resistant, reproducible benchmark.

- LC25000 images are assigned to **source-tile groups before splitting**; groups never cross train/validation/test.
- Five prespecified seeds (**42–46**) are required for the final summary.
- A fresh, identically seeded augmentation pipeline is created for each architecture so stochastic training views are comparable across models.
- LungHist700 is **external-only**: no retraining, fine-tuning, threshold tuning, or model selection uses it.
- The external analysis retains both image-level and patient-level predictions.
- The 12-tile/image assumption is explicitly checked before inference.
- The interval across the five seeds is descriptive; report **mean ± SD across seeds** and use cluster/patient bootstrap intervals for external evaluation.
- Keep the original image-level LC25000 result only as a historical benchmark; do not present it as leakage-free evidence of generalization.

**Scientific interpretation:** the group-aware rerun is valuable whether accuracy decreases, stays similar, or changes the model ranking. The experiment is not designed to manufacture a higher score.


In [ ]:
# %% Settings
import os
TEST_MODE = os.environ.get("GS_TEST_MODE") == "1"     # offline test only; leave as is

SEEDS           = [42, 43, 44, 45, 46]   # 5 repeated group-level splits (seed controls split, initialisation and shuffling)
MIN_SEEDS       = 5                      # require all five prespecified seeds for the final summary
IMG_SIZE        = 160
BATCH_SIZE      = 16
EPOCHS_STAGE1   = 8
EPOCHS_STAGE2   = 8
LR_STAGE1       = 1e-4
LR_STAGE2       = 5e-5
UNFREEZE_LAST_N = 40
PATIENCE        = 5
DROPOUT         = 0.3
WEIGHTS         = "imagenet"
TILE            = 400
N_BOOT          = 2000
GROUPS_COMMIT = "912f342fe2a781e847f395d79982dad4d3d425ae"      # pinned version of the LC25000-clean grouping file
GROUPS_SHA256 = "2ff9e992f759f265dfc9aa2363de0dfcc8e6559884d8bb02a4c925214fdf7e23"
GROUPS_URL = f"https://raw.githubusercontent.com/GeorgeBatch/LC25000-clean/{GROUPS_COMMIT}/kaggle/lc25000_image_groups.csv"
LH_URL     = "https://ndownloader.figshare.com/files/45206104"      # LungHist700 (figshare, CC BY 4.0)
SAVE_DIR   = "/content/drive/MyDrive/group_split_results"          # persistent: survives disconnects
DATA_DIR   = None
LH_ROOT    = "/content/lunghist700"
if TEST_MODE:
    EPOCHS_STAGE1, EPOCHS_STAGE2, WEIGHTS, N_BOOT, SEEDS, MIN_SEEDS = 1, 1, None, 100, [42, 43], 2
    DATA_DIR, LH_ROOT, SAVE_DIR = os.environ["GS_DATA"], os.environ["GS_LH"], os.environ["GS_OUT"]
    GROUPS_URL = os.environ["GS_GROUPS"]

In [ ]:
# %% Imports
import json, time, hashlib, random, glob, re, subprocess, shutil, zipfile
import numpy as np, pandas as pd, cv2
import matplotlib
if TEST_MODE: matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tensorflow as tf, keras
from keras import layers
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, balanced_accuracy_score
from statsmodels.stats.contingency_tables import mcnemar
keras.mixed_precision.set_global_policy("float32")
CLASSES = ["lung_aca", "lung_n", "lung_scc"]
NAMES = ["MobileNetV2", "EfficientNetB0", "Hybrid_EffNetB0_Transformer", "EffNetB0_TokenMLP"]   # last = parameter-matched ablation
if not TEST_MODE:
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(SAVE_DIR, exist_ok=True)
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))
print("Results are saved to:", SAVE_DIR)
import platform, subprocess as _sp
try:
    _gpu = _sp.run(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip() if not TEST_MODE else "test"
except FileNotFoundError:
    _gpu = "none (CPU runtime)"
json.dump({"python": platform.python_version(), "tensorflow": tf.__version__, "keras": keras.__version__,
           "cuda": tf.sysconfig.get_build_info().get("cuda_version"), "cudnn": tf.sysconfig.get_build_info().get("cudnn_version"),
           "gpu": _gpu, "seeds": SEEDS, "img_size": IMG_SIZE, "batch_size": BATCH_SIZE, "epochs": [EPOCHS_STAGE1, EPOCHS_STAGE2],
           "learning_rates": [LR_STAGE1, LR_STAGE2], "unfreeze_last_n": UNFREEZE_LAST_N, "patience": PATIENCE, "dropout": DROPOUT,
           "augmentation": "flip(h), rotation 15 deg, translation 0.1, zoom 0.1", "weights": WEIGHTS, "split": "source-group 70/15/15",
           "groups_url": GROUPS_URL, "groups_sha256": GROUPS_SHA256, "lc25000": "kaggle andrewmvd/lung-and-colon-cancer-histopathological-images",
           "lunghist700": LH_URL, "external_tiles": "12 x 400px, mean of tile probabilities, fixed before external testing",
           "models": NAMES}, open(os.path.join(SAVE_DIR, "config.json"), "w"), indent=1)

def seed_all(s):
    os.environ["PYTHONHASHSEED"] = str(s); random.seed(s); np.random.seed(s); tf.random.set_seed(s)

In [ ]:
# %% LC25000 lung data (MD5-deduplicated exactly as in the first benchmark) + source-tile groups
def find_lung_dir(root):
    for d, ds, _ in os.walk(root):
        if all(c in ds for c in CLASSES):
            return d
    raise FileNotFoundError(root)

if DATA_DIR is None:
    import kagglehub
    DATA_DIR = find_lung_dir(kagglehub.dataset_download("andrewmvd/lung-and-colon-cancer-histopathological-images"))
rows = []
for label, c in enumerate(CLASSES):
    for f in sorted(glob.glob(os.path.join(DATA_DIR, c, "*"))):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            with open(f, "rb") as fh:
                rows.append({"path": f, "label": label, "cls": c, "filename": os.path.basename(f),
                             "md5": hashlib.md5(fh.read()).hexdigest()})
df_all = pd.DataFrame(rows)
df = df_all.drop_duplicates("md5", keep="first").reset_index(drop=True)
conflict = df_all.groupby("md5")["label"].nunique()
df = df[~df["md5"].isin(conflict[conflict > 1].index)].reset_index(drop=True)

groups = pd.read_csv(GROUPS_URL)
if not TEST_MODE:
    import urllib.request
    assert hashlib.sha256(urllib.request.urlopen(GROUPS_URL).read()).hexdigest() == GROUPS_SHA256, "grouping file changed - send this message to Claude"
assert not groups.groupby("group_id")["label"].nunique().gt(1).any(), "a group_id occurs in more than one class"
assert groups["filename"].is_unique, "duplicate file names in grouping file"
groups = groups[groups["label"].isin(CLASSES)][["filename", "label", "group_id"]].rename(columns={"label": "g_label"})
df = df.merge(groups, on="filename", how="left")
assert df["group_id"].notna().all(), f"{df['group_id'].isna().sum()} images have no group - send this message to Claude"
assert (df["g_label"] == df["cls"]).all(), "group file labels do not match folder labels"
df["group_id"] = df["group_id"].astype(int)
print(f"Images: {len(df_all)} | exact duplicates removed: {len(df_all) - len(df)} | kept: {len(df)}")
print("Source-tile groups per class:", df.groupby("cls")["group_id"].nunique().to_dict())

# Leakage in the ORIGINAL image-level split (same code and seed as the first benchmark)
tr0, tmp0 = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=42)
va0, te0 = train_test_split(tmp0, test_size=0.50, stratify=tmp0["label"], random_state=42)
leak = {"test_images_with_group_in_train_pct": float(100 * te0["group_id"].isin(set(tr0["group_id"])).mean()),
        "test_groups_also_in_train_pct": float(100 * np.mean([g in set(tr0["group_id"]) for g in te0["group_id"].unique()]))}
print("Original image-level split:", leak)

def group_split(df, seed, fr=(0.70, 0.15, 0.15)):
    """Per class, shuffle the source groups, then place each whole group in the partition that is furthest
    below its target number of images (70/15/15). Groups are never divided."""
    rng = np.random.default_rng(seed); assign = {}; keys = ["train", "val", "test"]
    for c in CLASSES:
        sizes = df[df["cls"] == c].groupby("group_id").size()
        g = sizes.index.values.copy(); rng.shuffle(g)
        target = np.array(fr) * sizes.sum(); filled = np.zeros(3)
        for gid in g:
            k = int(np.argmax((target - filled) / target)); assign[gid] = keys[k]; filled[k] += sizes[gid]
    s = df.assign(split=df["group_id"].map(assign))
    parts = {k: s[s["split"] == k].reset_index(drop=True) for k in ["train", "val", "test"]}
    for a in parts:
        for b in parts:
            if a < b:
                assert not set(parts[a]["group_id"]) & set(parts[b]["group_id"]), "group overlap!"
                assert not set(parts[a]["md5"]) & set(parts[b]["md5"]), "hash overlap!"
    return parts

In [ ]:
# %% Shared pipeline and models (identical to the first benchmark)
AUTOTUNE = tf.data.AUTOTUNE
def make_augment(seed):
    # A fresh pipeline is created for each model/seed so augmentation RNG state
    # does not advance from one architecture to another.
    return keras.Sequential([
        layers.RandomFlip("horizontal", seed=seed + 101),
        layers.RandomRotation(15 / 360, seed=seed + 102),
        layers.RandomTranslation(0.1, 0.1, seed=seed + 103),
        layers.RandomZoom(0.1, seed=seed + 104),
    ], name=f"augment_seed{seed}")

def load(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return tf.cast(tf.round(img), tf.uint8), tf.one_hot(label, len(CLASSES))

def make_ds(part, training, seed):
    ds = tf.data.Dataset.from_tensor_slices((part["path"].values, part["label"].values))
    ds = ds.map(load, num_parallel_calls=AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(len(part), seed=seed, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE).map(
        lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE
    )
    if training:
        aug = make_augment(seed)
        ds = ds.map(lambda x, y: (aug(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

def head(x):
    x = layers.Dropout(DROPOUT, name="dropout")(x)
    return layers.Activation("softmax", name="pred")(layers.Dense(len(CLASSES), name="logits")(x))

def build_mobilenet():
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    x = layers.Rescaling(1 / 127.5, offset=-1, name="mnv2_preprocess")(inp)
    base = keras.applications.MobileNetV2(include_top=False, weights=WEIGHTS, input_tensor=x)
    return keras.Model(inp, head(layers.GlobalAveragePooling2D(name="gap")(base.output)), name="MobileNetV2"), base

def build_efficientnet():
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    base = keras.applications.EfficientNetB0(include_top=False, weights=WEIGHTS, input_tensor=inp)
    return keras.Model(inp, head(layers.GlobalAveragePooling2D(name="gap")(base.output)), name="EfficientNetB0"), base

def build_hybrid():
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    base = keras.applications.EfficientNetB0(include_top=False, weights=WEIGHTS, input_tensor=inp)
    fmap = base.output; h, w, c = fmap.shape[1], fmap.shape[2], fmap.shape[3]
    tokens = layers.Reshape((h * w, c), name="tokens")(fmap)
    t = layers.LayerNormalization(epsilon=1e-6, name="ln1")(tokens)
    attn_out = layers.MultiHeadAttention(num_heads=2, key_dim=16, name="mhsa")(t, t)
    x = layers.Add(name="res1")([tokens, attn_out])
    t = layers.LayerNormalization(epsilon=1e-6, name="ln2")(x)
    t = layers.Dense(c, name="mlp2")(layers.Dense(128, activation="gelu", name="mlp1")(t))
    x = layers.GlobalAveragePooling1D(name="gap")(layers.Add(name="res2")([x, t]))
    return keras.Model(inp, head(x), name="Hybrid_EffNetB0_Transformer"), base

def build_tokenmlp():
    """Ablation: the hybrid block with self-attention removed and the token MLP widened (hidden 192) so that the
    added parameters match the transformer block (~0.50M). Tests attention itself versus extra capacity."""
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    base = keras.applications.EfficientNetB0(include_top=False, weights=WEIGHTS, input_tensor=inp)
    fmap = base.output; h, w, c = fmap.shape[1], fmap.shape[2], fmap.shape[3]
    tokens = layers.Reshape((h * w, c), name="tokens")(fmap)
    t = layers.LayerNormalization(epsilon=1e-6, name="ln2")(tokens)
    t = layers.Dense(c, name="mlp2")(layers.Dense(192, activation="gelu", name="mlp1")(t))
    x = layers.GlobalAveragePooling1D(name="gap")(layers.Add(name="res2")([tokens, t]))
    return keras.Model(inp, head(x), name="EffNetB0_TokenMLP"), base

BUILDERS = {"MobileNetV2": build_mobilenet, "EfficientNetB0": build_efficientnet,
            "Hybrid_EffNetB0_Transformer": build_hybrid, "EffNetB0_TokenMLP": build_tokenmlp}

def set_stage2(base):
    base.trainable = True
    for l in base.layers[:-UNFREEZE_LAST_N]: l.trainable = False
    for l in base.layers:
        if isinstance(l, layers.BatchNormalization): l.trainable = False

def callbacks(ckpt):
    return [keras.callbacks.ModelCheckpoint(ckpt, monitor="val_accuracy", save_best_only=True, save_weights_only=True),
            keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=PATIENCE, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7)]

def ece_score(p, y, n_bins=15):
    conf, pred = p.max(1), p.argmax(1); bins, e = np.linspace(0, 1, n_bins + 1), 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(e)

def cluster_ci(correct, clusters, seed=0):
    """Bootstrap that resamples clusters (source groups or patients) with replacement, keeping all their images."""
    rng = np.random.default_rng(seed); u = np.unique(clusters); idx = {k: np.where(clusters == k)[0] for k in u}; a = []
    for _ in range(N_BOOT):
        s = np.concatenate([idx[k] for k in rng.choice(u, len(u), replace=True)]); a.append(correct[s].mean())
    return [float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))]

def metrics(p, y, clusters):
    yp = p.argmax(1); cm = confusion_matrix(y, yp, labels=[0, 1, 2])
    rec = cm.diagonal() / cm.sum(1).clip(min=1)
    return {"n": int(len(y)), "accuracy": float((yp == y).mean()),
            "accuracy_95ci_cluster": cluster_ci((yp == y).astype(float), clusters),
            "macro_f1": float(f1_score(y, yp, average="macro")), "balanced_accuracy": float(balanced_accuracy_score(y, yp)),
            "ece": ece_score(p, y), "brier": float(((p - np.eye(3)[y]) ** 2).sum(1).mean()),
            "recall": dict(zip(CLASSES, rec.round(4).tolist())),
            "auc_ovr_macro": float(roc_auc_score(y, p, multi_class="ovr")) if len(np.unique(y)) == 3 else None,
            "errors": int((yp != y).sum()), "confusion_matrix": cm.tolist()}

In [ ]:
# %% Train all models for every seed (skips any run already saved on Drive)
for seed in SEEDS:
    parts = group_split(df, seed)
    sdir = os.path.join(SAVE_DIR, f"seed{seed}"); os.makedirs(sdir, exist_ok=True)
    for k, p in parts.items():
        p.drop(columns=["g_label"]).to_csv(os.path.join(sdir, f"split_{k}.csv"), index=False)
    pd.DataFrame([{"seed": seed, "class": c, "split": k, "images": int((p["cls"] == c).sum()),
                   "groups": int(p.loc[p["cls"] == c, "group_id"].nunique())} for k, p in parts.items() for c in CLASSES]
                 ).to_csv(os.path.join(sdir, "split_balance.csv"), index=False)
    print(f"\nSEED {seed}: " + " | ".join(f"{k} {len(p)} images ({100*len(p)/len(df):.1f}%) / {p['group_id'].nunique()} groups" for k, p in parts.items()))
    ds_cache = {}
    for name in NAMES:
        mdir = os.path.join(sdir, name); done = os.path.join(mdir, "internal.json")
        if os.path.exists(done):
            print(f"  {name}: already finished - skipped"); continue
        # Fresh pipelines are intentional: each architecture receives the same
        # seeded stochastic training views rather than inheriting RNG state from
        # a preceding architecture.
        print("=" * 70 + f"\nSEED {seed} - TRAINING {name}\n" + "=" * 70)
        os.makedirs(mdir, exist_ok=True); keras.backend.clear_session(); seed_all(seed)
        ds_cache = {
            "train": make_ds(parts["train"], True, seed),
            "val": make_ds(parts["val"], False, seed),
            "test": make_ds(parts["test"], False, seed)
        }
        model, base = BUILDERS[name](); ckpt = os.path.join(mdir, "best.weights.h5"); t0 = time.time()
        base.trainable = False
        model.compile(keras.optimizers.Adam(LR_STAGE1), "categorical_crossentropy", metrics=["accuracy"])
        tr1 = int(sum(np.prod(w.shape) for w in model.trainable_weights))
        h1 = model.fit(ds_cache["train"], validation_data=ds_cache["val"], epochs=EPOCHS_STAGE1, callbacks=callbacks(ckpt), verbose=2)
        set_stage2(base)
        model.compile(keras.optimizers.Adam(LR_STAGE2), "categorical_crossentropy", metrics=["accuracy"])
        h2 = model.fit(ds_cache["train"], validation_data=ds_cache["val"], epochs=EPOCHS_STAGE2, callbacks=callbacks(ckpt), verbose=2)
        tr2 = int(sum(np.prod(w.shape) for w in model.trainable_weights)); model.load_weights(ckpt)
        hist = {k: [float(v) for v in h1.history[k] + h2.history[k]] for k in h1.history if k in h2.history}
        hist["stage1_epochs"] = len(h1.history["loss"])
        p = model.predict(ds_cache["test"], verbose=0); y = parts["test"]["label"].values
        res = metrics(p, y, parts["test"]["group_id"].values)
        res.update({"seed": seed, "model": name, "minutes": round((time.time() - t0) / 60, 1),
                    "params": {"total": int(model.count_params()), "trainable_stage1": tr1, "trainable_stage2": tr2},
                    "best_val_accuracy": float(max(hist["val_accuracy"])),
                    "best_val_accuracy_stage1": float(max(hist["val_accuracy"][:hist["stage1_epochs"]]))})
        pd.DataFrame({"path": parts["test"]["path"], "group_id": parts["test"]["group_id"], "y_true": y, "y_pred": p.argmax(1),
                      **{f"p_{c}": p[:, i] for i, c in enumerate(CLASSES)}}).to_csv(os.path.join(mdir, "test_predictions.csv"), index=False)
        json.dump(hist, open(os.path.join(mdir, "history.json"), "w"))
        json.dump(res, open(done, "w"), indent=1)
        print(f"  -> {name} seed {seed}: accuracy {100*res['accuracy']:.2f}% | macro-F1 {res['macro_f1']:.4f} | "
              f"balanced {100*res['balanced_accuracy']:.2f}% | ECE {res['ece']:.4f} | {res['minutes']} min")
        del model

In [ ]:
# %% LungHist700: download once, cut every image into 12 tiles of 400x400 (no colour normalisation)
if not glob.glob(LH_ROOT + "/**/*.jpg", recursive=True):
    os.makedirs(LH_ROOT, exist_ok=True); rar = "/content/LungHist700.rar"
    if not os.path.exists(rar): subprocess.run(["wget", "-q", "-O", rar, LH_URL], check=True)
    with open(rar, "rb") as fh: lh_sha = hashlib.sha256(fh.read()).hexdigest()
    json.dump({"figshare_file_id": "45206104", "url": LH_URL, "sha256": lh_sha, "downloaded": time.strftime("%Y-%m-%d")},
              open(os.path.join(SAVE_DIR, "lunghist700_download.json"), "w"), indent=1)
    if subprocess.run(["which", "unrar"], capture_output=True).returncode != 0:
        subprocess.run("apt-get -qq install -y unrar > /dev/null", shell=True)
    if subprocess.run(["unrar", "x", "-o+", "-inul", rar, LH_ROOT + "/"]).returncode != 0:
        subprocess.run(["7z", "x", "-y", f"-o{LH_ROOT}", rar], check=True, stdout=subprocess.DEVNULL)
rows = []
for f in sorted(glob.glob(LH_ROOT + "/**/*.jp*g", recursive=True)):
    m = re.match(r"^(aca|nor|scc)(?:_(bd|md|pd))?_(20x|40x)_(\d+)\.jpe?g$", os.path.basename(f).lower())
    if m:
        sup, sub, res_, iid = m.groups()
        rows.append({"path": f, "superclass": sup, "subclass": sub or "", "resolution": res_, "image_id": int(iid),
                     "label": {"aca": 0, "nor": 1, "scc": 2}[sup]})
ext = pd.DataFrame(rows)
for c in glob.glob(LH_ROOT + "/**/*.csv", recursive=True):
    meta = pd.read_csv(c)
    if {"superclass", "resolution", "image_id", "patient_id"} <= set(meta.columns):
        meta = meta.copy()
        if "subclass" not in meta: meta["subclass"] = ""
        meta["subclass"] = meta["subclass"].fillna("").astype(str).str.lower().replace("nan", "")
        meta["superclass"] = meta["superclass"].astype(str).str.lower(); meta["resolution"] = meta["resolution"].astype(str).str.lower()
        meta["image_id"] = meta["image_id"].astype(int)
        ext = ext.merge(meta[["superclass", "subclass", "resolution", "image_id", "patient_id"]],
                        on=["superclass", "subclass", "resolution", "image_id"], how="left"); break
assert "patient_id" in ext and ext["patient_id"].notna().all(), "patient IDs missing - send this message to Claude"
print(f"LungHist700: {len(ext)} images, {ext['patient_id'].nunique()} patients")
pat_table = {"images": ext.groupby("superclass").size().to_dict(),
             "patients": ext.groupby("superclass")["patient_id"].nunique().to_dict(),
             "patients_in_more_than_one_class": int((ext.groupby("patient_id")["superclass"].nunique() > 1).sum()),
             "images_per_patient": ext.groupby("patient_id").size().describe().round(1).to_dict(),
             "patients_by_resolution": ext.groupby("resolution")["patient_id"].nunique().to_dict()}
print(json.dumps(pat_table, indent=1))
ext.to_csv(os.path.join(SAVE_DIR, "lunghist700_index.csv"), index=False)

tiles, sizes_seen, tile_counts = [], set(), []
for f in ext["path"]:
    img = cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB); H, W = img.shape[:2]; sizes_seen.add((H, W)); n0 = len(tiles)
    for yy in range(0, H - TILE + 1, TILE):
        for xx in range(0, W - TILE + 1, TILE):
            tiles.append(cv2.resize(img[yy:yy + TILE, xx:xx + TILE], (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA))
    n_tiles = len(tiles) - n0
    tile_counts.append(n_tiles)
    assert n_tiles == 12, f"{f}: {H}x{W} gives {n_tiles} tiles instead of 12 - send this message to Claude"
assert len(set(tile_counts)) == 1 and tile_counts[0] == 12, "not every image has exactly 12 tiles"
print("LungHist700 image sizes found:", sizes_seen)
tiles = np.stack(tiles).astype(np.uint8); T = 12
print(f"{len(tiles)} tiles ({T} per image)")
y_ext, pat = ext["label"].values, ext["patient_id"].values

In [ ]:
# %% External test of every trained model (no retraining, no tuning)
for seed in SEEDS:
    for name in NAMES:
        mdir = os.path.join(SAVE_DIR, f"seed{seed}", name); done = os.path.join(mdir, "external.json")
        if os.path.exists(done): continue
        keras.backend.clear_session(); model, _ = BUILDERS[name](); model.load_weights(os.path.join(mdir, "best.weights.h5"))
        pt = model.predict(tiles.astype(np.float32), batch_size=64, verbose=0)
        p = pt.reshape(len(ext), T, 3).mean(1)
        res = metrics(p, y_ext, pat); res["by_resolution"] = {r: float((p.argmax(1)[ext["resolution"] == r] == y_ext[ext["resolution"] == r]).mean())
                                                            for r in ["20x", "40x"]}
        np.save(os.path.join(mdir, "external_probs.npy"), p); json.dump(res, open(done, "w"), indent=1)
        image_pred = ext[["patient_id", "superclass", "subclass", "resolution", "image_id", "label"]].rename(
            columns={"label": "y_true"}).assign(
                y_pred=p.argmax(1), **{f"p_{c}": p[:, i] for i, c in enumerate(CLASSES)})
        image_pred.to_csv(os.path.join(mdir, "external_predictions.csv"), index=False)

        # Secondary patient-level aggregation; never used for model selection/tuning.
        # LungHist700 patients contribute images of several classes (normal tissue comes from the same cancer
        # patients), so probabilities are averaged per patient AND true class ("patient-class units").
        patient_pred = image_pred.groupby(["patient_id", "y_true"], as_index=False).agg(
            n_images=("image_id", "size"),
            p_lung_aca=("p_lung_aca", "mean"),
            p_lung_n=("p_lung_n", "mean"),
            p_lung_scc=("p_lung_scc", "mean"))
        patient_pred["y_pred"] = patient_pred[
            ["p_lung_aca", "p_lung_n", "p_lung_scc"]
        ].to_numpy().argmax(1)
        patient_pred.to_csv(
            os.path.join(mdir, "external_patient_predictions.csv"), index=False)
        print(f"{name} seed {seed}: external accuracy {100*res['accuracy']:.1f}% | balanced {100*res['balanced_accuracy']:.1f}%")
        del model

In [ ]:
# %% Aggregate across seeds: mean ± SD, 95% CI (t-distribution over seeds), paired comparisons
def agg(vals):
    v = np.array(vals, float); m, sd = v.mean(), v.std(ddof=1) if len(v) > 1 else 0.0
    h = stats.t.ppf(0.975, len(v) - 1) * sd / np.sqrt(len(v)) if len(v) > 1 else 0.0
    return {"mean": float(m), "sd": float(sd), "ci95": [float(m - h), float(m + h)], "per_seed": v.round(5).tolist()}

SEEDS_ALL = SEEDS
SEEDS = [s for s in SEEDS_ALL if all(os.path.exists(os.path.join(SAVE_DIR, f"seed{s}", n, "external.json")) for n in NAMES)]
assert len(SEEDS) >= min(MIN_SEEDS, len(SEEDS_ALL)), f"only {len(SEEDS)} seeds complete - run all cells again to continue"
print("Seeds included in the summary:", SEEDS)
R = {s: {n: {"int": json.load(open(os.path.join(SAVE_DIR, f"seed{s}", n, "internal.json"))),
             "ext": json.load(open(os.path.join(SAVE_DIR, f"seed{s}", n, "external.json")))} for n in NAMES} for s in SEEDS}
summary = {"seeds": SEEDS, "leakage_original_split": leak, "lunghist700_patients": pat_table, "models": {}, "paired": {}}
for n in NAMES:
    summary["models"][n] = {
        f"{d}_{k}": agg([R[s][n][d][k] for s in SEEDS])
        for d in ["int", "ext"] for k in ["accuracy", "macro_f1", "balanced_accuracy", "ece", "brier"]}
    summary["models"][n]["int_best_val_accuracy_stage1"] = agg([R[s][n]["int"]["best_val_accuracy_stage1"] for s in SEEDS])
    summary["models"][n]["ext_accuracy_20x"] = agg([R[s][n]["ext"]["by_resolution"]["20x"] for s in SEEDS])
    summary["models"][n]["ext_accuracy_40x"] = agg([R[s][n]["ext"]["by_resolution"]["40x"] for s in SEEDS])
    summary["models"][n]["int_recall"] = {c: agg([R[s][n]["int"]["recall"][c] for s in SEEDS]) for c in CLASSES}
    summary["models"][n]["ext_recall"] = {c: agg([R[s][n]["ext"]["recall"][c] for s in SEEDS]) for c in CLASSES}
    summary["models"][n]["int_errors"] = [R[s][n]["int"]["errors"] for s in SEEDS]
    summary["models"][n]["int_test_n"] = [R[s][n]["int"]["n"] for s in SEEDS]

pairs = [("Hybrid_EffNetB0_Transformer", "MobileNetV2"), ("Hybrid_EffNetB0_Transformer", "EfficientNetB0"), ("EfficientNetB0", "MobileNetV2"),
         ("Hybrid_EffNetB0_Transformer", "EffNetB0_TokenMLP"), ("EffNetB0_TokenMLP", "EfficientNetB0")]
for a, b in pairs:
    key = f"{a} - {b}"; summary["paired"][key] = {"internal_mcnemar_per_seed": [], "ext_paired_patient_bootstrap_per_seed": []}
    for s in SEEDS:
        ta = pd.read_csv(os.path.join(SAVE_DIR, f"seed{s}", a, "test_predictions.csv"))
        tb = pd.read_csv(os.path.join(SAVE_DIR, f"seed{s}", b, "test_predictions.csv"))
        ca, cb = (ta.y_pred == ta.y_true).values, (tb.y_pred == tb.y_true).values
        tab = [[int((ca & cb).sum()), int((ca & ~cb).sum())], [int((~ca & cb).sum()), int((~ca & ~cb).sum())]]
        summary["paired"][key]["internal_mcnemar_per_seed"].append(
            {"seed": s, "diff_pp": float(100 * (ca.mean() - cb.mean())), "only_a": tab[0][1], "only_b": tab[1][0],
             "p": float(mcnemar(tab, exact=True).pvalue)})
        pa = np.load(os.path.join(SAVE_DIR, f"seed{s}", a, "external_probs.npy")).argmax(1) == y_ext
        pb = np.load(os.path.join(SAVE_DIR, f"seed{s}", b, "external_probs.npy")).argmax(1) == y_ext
        d = (pa.astype(float) - pb.astype(float)); ci = cluster_ci(d, pat)
        summary["paired"][key]["ext_paired_patient_bootstrap_per_seed"].append(
            {"seed": s, "diff_pp": float(100 * d.mean()), "ci95_pp": [100 * ci[0], 100 * ci[1]]})
    summary["paired"][key]["internal_diff_pp"] = agg([x["diff_pp"] for x in summary["paired"][key]["internal_mcnemar_per_seed"]])
    summary["paired"][key]["external_diff_pp"] = agg([x["diff_pp"] for x in summary["paired"][key]["ext_paired_patient_bootstrap_per_seed"]])
ORIGINAL = {"MobileNetV2": 0.9878, "EfficientNetB0": 0.9901, "Hybrid_EffNetB0_Transformer": 0.9944}   # image-level split, seed 42
summary["delta_vs_original_split_pp"] = {n: {"original": 100 * ORIGINAL[n],
    "group_split_mean": 100 * summary["models"][n]["int_accuracy"]["mean"],
    "delta": 100 * (summary["models"][n]["int_accuracy"]["mean"] - ORIGINAL[n])} for n in NAMES if n in ORIGINAL}
summary["complete"] = len(SEEDS) == len(SEEDS_ALL)
summary["split_proportions"] = {s: {k: len(pd.read_csv(os.path.join(SAVE_DIR, f"seed{s}", f"split_{k}.csv"))) for k in ["train", "val", "test"]} for s in SEEDS}
summary["note"] = "ci95 over seeds uses a t-distribution with n-1 df and is descriptive only; report mean +/- SD across splits"
json.dump(summary, open(os.path.join(SAVE_DIR, "group_split_summary.json"), "w"), indent=1)

In [ ]:
# %% Figure: internal (group split) vs external accuracy per seed
fig, ax = plt.subplots(figsize=(8.5, 4.2)); short = {"MobileNetV2": "MobileNetV2", "EfficientNetB0": "EfficientNetB0", "Hybrid_EffNetB0_Transformer": "Hybrid", "EffNetB0_TokenMLP": "TokenMLP (ablation)"}
for i, n in enumerate(NAMES):
    for j, (d, mk) in enumerate([("int", "o"), ("ext", "s")]):
        v = 100 * np.array(summary["models"][n][f"{d}_accuracy"]["per_seed"])
        ax.scatter(np.full(len(v), i + (j - 0.5) * 0.25), v, marker=mk, color=["#1f4e79", "#e69f00"][j],
                   label=["LC25000 group split", "LungHist700"][j] if i == 0 else None, zorder=3)
        ax.errorbar(i + (j - 0.5) * 0.25, v.mean(), yerr=v.std(ddof=1) if len(v) > 1 else 0, color="black", capsize=4, zorder=2)
ax.axhline(100 * 280 / 691, ls="--", c="gray", lw=1, label="LungHist700 majority class")
ax.set_xticks(range(len(NAMES)), [short[n] for n in NAMES]); ax.set_ylabel("Accuracy (%)"); ax.legend(fontsize=8, loc="center right")
ax.set_title(f"Per-seed accuracy (n = {len(SEEDS)} seeds)"); fig.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, "fig_group_split_accuracy.png"), dpi=300); plt.show() if not TEST_MODE else None; plt.close(fig)

In [ ]:
# %% SUMMARY (copy everything below and send it to Claude together with group_split_results.zip)
def f(a, scale=100, d=2): return f"{scale*a['mean']:.{d}f} ± {scale*a['sd']:.{d}f}"
print("=" * 78 + "\nSUMMARY - source-group LC25000 split, seeds", SEEDS)
if not summary["complete"]:
    print(f"*** PROVISIONAL: only {len(SEEDS)} of {len(SEEDS_ALL)} seeds are complete. Run all cells again to finish the remaining seeds. ***")
print("Original image-level split leakage:", leak)
print("\nModel           | Internal acc (%) | Int macro-F1 | Int bal acc | Int ECE | External acc (%) | Ext bal acc | Ext ECE")
for n in NAMES:
    m = summary["models"][n]
    print(f"{short[n]:15s} | {f(m['int_accuracy'])} | {f(m['int_macro_f1'],1,4)} | {f(m['int_balanced_accuracy'])} | "
          f"{f(m['int_ece'],1,4)} | {f(m['ext_accuracy'],100,1)} | {f(m['ext_balanced_accuracy'],100,1)} | {f(m['ext_ece'],1,3)}")
print("\nPaired differences (percentage points, mean ± SD over seeds):")
for k, v in summary["paired"].items():
    print(f"  {k}: internal {v['internal_diff_pp']['mean']:+.2f} ± {v['internal_diff_pp']['sd']:.2f} "
          f"(McNemar p per seed: {[round(x['p'], 4) for x in v['internal_mcnemar_per_seed']]}) | "
          f"external {v['external_diff_pp']['mean']:+.1f} ± {v['external_diff_pp']['sd']:.1f}")
print("\nOriginal image-level split vs group split (internal accuracy, %):")
for n in summary["delta_vs_original_split_pp"]:
    d = summary["delta_vs_original_split_pp"][n]
    print(f"  {short[n]:15s} {d['original']:.2f} -> {d['group_split_mean']:.2f}  (change {d['delta']:+.2f})")
print("\nLungHist700 patients:", json.dumps(pat_table))
print(json.dumps(summary, indent=None)[:6000])

zpath = "/content/group_split_results.zip" if not TEST_MODE else os.path.join(SAVE_DIR, "..", "group_split_results.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for p_ in glob.glob(SAVE_DIR + "/**/*", recursive=True):
        if os.path.isfile(p_) and not p_.endswith(".h5"):
            z.write(p_, os.path.relpath(p_, SAVE_DIR))
print("Created", zpath, "(model weights stay on Google Drive in", SAVE_DIR + ")")
if not TEST_MODE:
    from google.colab import files; files.download(zpath)